In [1]:
import pandas as pd

In [8]:
df = pd.read_csv(r"C:\Users\Manya Ali\Desktop\manya ali 33142\teaching_prices_cleaned.xls")

In [9]:
df.head()

,date,item,city,price,unit,data_label,item_original,price_numeric,price_flag,price_clean,key_conflict
0,2026-01-01,Staple A,City 1,101.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...,Staple A,101.0,OK,101.0,False
1,2026-01-01,Staple A,City 2,107.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...,Staple A,107.0,OK,107.0,False
2,2026-01-01,Oil B,City 1,298.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...,Oil B,298.0,OK,298.0,False
3,2026-01-01,Oil B,City 2,304.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...,Oil B,304.0,OK,304.0,False
4,2026-01-01,Pulse C,City 1,200.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...,Pulse C,200.0,OK,200.0,False


In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           96 non-null     str    
 1   item           96 non-null     str    
 2   city           96 non-null     str    
 3   price          95 non-null     str    
 4   unit           96 non-null     str    
 5   data_label     96 non-null     str    
 6   item_original  96 non-null     str    
 7   price_numeric  95 non-null     float64
 8   price_flag     96 non-null     str    
 9   price_clean    93 non-null     float64
 10  key_conflict   96 non-null     bool   
dtypes: bool(1), float64(2), str(8)
memory usage: 19.0 KB


In [16]:
item_statistics = df.groupby("item")["price"].agg(
    Mean="mean",
    Median="median",
    Count="count"
).reset_index()

print(item_statistics)

TypeError: dtype 'str' does not support operation 'mean'

In [17]:
print(df["price"].head(20))
print(df["price"].dtype)

0     101.0
1     107.0
2     298.0
3     304.0
4     200.0
5       NaN
6     149.0
7     155.0
8      99.5
9     105.5
10    304.0
11    310.0
12    202.5
13    208.5
14    153.5
15    159.5
16    103.0
17    1,250
18    310.0
19    316.0
Name: price, dtype: str
str


In [18]:
df["price"] = pd.to_numeric(df["price"], errors="coerce")

In [19]:
print(df["price"].dtype)
print(df["price"].isna().sum())

float64
2


In [20]:
item_statistics = df.groupby("item")["price"].agg(
    Mean="mean",
    Median="median",
    Count="count"
).reset_index()

print(item_statistics)

          item        Mean  Median  Count
0        Oil B  316.180000  324.00     25
1      Pulse C  205.913043  206.00     23
2     Staple A  111.386364  112.50     22
3  Sweetener D  576.812500  167.75     24


In [21]:
df["date"] = pd.to_datetime(df["date"])

df = df.sort_values("date")

first_last = df.groupby("item").agg(
    First_Price=("price", "first"),
    Last_Price=("price", "last")
).reset_index()

first_last["Percentage_Change"] = (
    (first_last["Last_Price"] - first_last["First_Price"])
    / first_last["First_Price"]
) * 100

print(first_last)

          item  First_Price  Last_Price  Percentage_Change
0        Oil B        298.0       350.0          17.449664
1      Pulse C        200.0       213.5           6.750000
2     Staple A        101.0       120.5          19.306931
3  Sweetener D        149.0       184.5          23.825503


In [22]:
final_statistics = item_statistics.merge(
    first_last,
    on="item"
)

print(final_statistics.round(2))

          item    Mean  Median  Count  First_Price  Last_Price  \
0        Oil B  316.18  324.00     25        298.0       350.0   
1      Pulse C  205.91  206.00     23        200.0       213.5   
2     Staple A  111.39  112.50     22        101.0       120.5   
3  Sweetener D  576.81  167.75     24        149.0       184.5   

   Percentage_Change  
0              17.45  
1               6.75  
2              19.31  
3              23.83  


In [26]:
print(df.columns.tolist())

['date', 'item', 'city', 'price', 'unit', 'data_label', 'item_original', 'price_numeric', 'price_flag', 'price_clean', 'key_conflict']


In [28]:
print(df["price_flag"].value_counts(dropna=False))

display(
    df[df["price_flag"].notna()][
        ["date", "item", "price_numeric", "price_flag", "price_clean"]
    ].head(20)
)

price_flag
OK                       93
Suspicious high value     2
Missing or invalid        1
Name: count, dtype: int64


,date,item,price_numeric,price_flag,price_clean
0,2026-01-01,Staple A,101.0,OK,101.0
1,2026-01-01,Staple A,107.0,OK,107.0
2,2026-01-01,Oil B,298.0,OK,298.0
3,2026-01-01,Oil B,304.0,OK,304.0
4,2026-01-01,Pulse C,200.0,OK,200.0
5,2026-01-01,Pulse C,NaN,Missing or invalid,NaN
6,2026-01-01,Sweetener D,149.0,OK,149.0
7,2026-01-01,Sweetener D,155.0,OK,155.0
15,2026-01-08,Sweetener D,159.5,OK,159.5
14,2026-01-08,Sweetener D,153.5,OK,153.5


In [29]:
flagged_rows = df[df["price_flag"].notna()]

print(flagged_rows["item"].value_counts())

item
Oil B          25
Pulse C        24
Sweetener D    24
Staple A       23
Name: count, dtype: int64


In [30]:
# Select an item
item_name = "Rice"

item_data = df[df["item"] == item_name].copy()

# Convert price to numeric
item_data["price_numeric"] = pd.to_numeric(
    item_data["price_numeric"], errors="coerce"
)

# Statistics WITH flagged observations
mean_with = item_data["price_numeric"].mean()
median_with = item_data["price_numeric"].median()

# Statistics WITHOUT flagged observations
item_without_flag = item_data[
    item_data["price_flag"].isna()
]

mean_without = item_without_flag["price_numeric"].mean()
median_without = item_without_flag["price_numeric"].median()

print("WITH FLAGGED VALUE")
print("Mean:", mean_with)
print("Median:", median_with)

print("\nWITHOUT FLAGGED VALUE")
print("Mean:", mean_without)
print("Median:", median_without)

WITH FLAGGED VALUE
Mean: nan
Median: nan

WITHOUT FLAGGED VALUE
Mean: nan
Median: nan


In [31]:
comparison = pd.DataFrame({
    "Statistic": ["Mean", "Median"],
    "With Flagged Value": [mean_with, median_with],
    "Without Flagged Value": [mean_without, median_without]
})

comparison["Difference"] = (
    comparison["With Flagged Value"]
    - comparison["Without Flagged Value"]
)

display(comparison.round(2))

,Statistic,With Flagged Value,Without Flagged Value,Difference
0,Mean,NaN,NaN,NaN
1,Median,NaN,NaN,NaN
